In [1]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.8-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatib

In [2]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail \
     --retry 999 \
     --retry-all-errors \
     --retry-delay 5 \
     --retry-max-time 600 \
     http://gateway:8001/api/games


In [3]:
import os
os.environ["AGI_ARC_REPO"] = "/kaggle/input/datasets/suminshim/agi-arc-submit/agi-arc-main"

In [4]:
import os
import subprocess
import sys
from pathlib import Path


def find_agi_arc_repo() -> Path:
    candidates = []
    env_repo = os.getenv('AGI_ARC_REPO')
    if env_repo:
        candidates.append(Path(env_repo))
    input_root = Path('/kaggle/input')
    if input_root.exists():
        candidates.extend(sorted(path for path in input_root.iterdir() if path.is_dir()))
    for candidate in candidates:
        if (candidate / 'src' / 'arc_agi3').exists() and (candidate / 'scripts' / 'kaggle_competition_submission.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the agi-arc repo in /kaggle/input. Upload this repo as a Kaggle Dataset input.')


if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    COMPETITION_ROOT = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3'
    REPO_SOURCE = find_agi_arc_repo()
    WORKING_COPY = '/kaggle/working/ARC-AGI-3-Agents-agi-arc-submission'
    os.environ['AGI_ARC_REPO'] = str(REPO_SOURCE)
    os.environ.setdefault('ARC_AGI3_MAX_STEPS', '128')
    os.environ.setdefault('ARC_AGI3_EXPLORE_STEPS', '24')
    print(f'Using agi-arc source from input: {REPO_SOURCE}')
    print(f'Writable execution copy will be created at: {WORKING_COPY}')
    subprocess.run(
        [
            sys.executable,
            str(REPO_SOURCE / 'scripts' / 'kaggle_competition_submission.py'),
            '--repo-root',
            str(REPO_SOURCE),
            '--competition-root',
            COMPETITION_ROOT,
            '--working-root',
            '/kaggle/working',
            '--description',
            'ARC-AGI-3-Agents-agi-arc-submission',
        ],
        check=True,
    )


In [5]:
# Non-rerun mode: produce a dummy submission so the notebook
# has valid output for the initial commit / save.
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    pd.DataFrame({'task_id': ['dummy'], 'output': ['[[0]]']}).to_parquet(
        'submission.parquet', index=False)
    print('Non-competition mode: wrote dummy submission.parquet')


Non-competition mode: wrote dummy submission.parquet
